# 17.2 - Deep Learning Capstone

Status: VERIFIED

## What Are We Solving?

Build a small neural network from scratch in PyTorch. We generate synthetic regression data, define a simple `nn.Module`, train with a loop, plot loss curves, and evaluate.

In [1]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

np.random.seed(0)
torch.manual_seed(0)

N = 300
x_np = np.linspace(-3, 3, N).astype(np.float32)
y_np = np.sin(x_np) + 0.3 * np.random.randn(N).astype(np.float32)

split = int(0.8 * N)
x_train_t = torch.tensor(x_np[:split]).unsqueeze(1)
y_train_t = torch.tensor(y_np[:split]).unsqueeze(1)
x_test_t = torch.tensor(x_np[split:]).unsqueeze(1)
y_test_t = torch.tensor(y_np[split:]).unsqueeze(1)

train_ds = TensorDataset(x_train_t, y_train_t)
train_dl = DataLoader(train_ds, batch_size=32, shuffle=True)
print(f'Train: {len(train_ds)} | Test: {len(x_test_t)}')

Train: 240 | Test: 60


In [2]:
class SimpleNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 32), nn.ReLU(),
            nn.Linear(32, 32), nn.ReLU(),
            nn.Linear(32, 1),
        )
    def forward(self, x):
        return self.net(x)

model = SimpleNet()
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)
print(f'Parameters: {sum(p.numel() for p in model.parameters())}')
print(model)

Parameters: 1153
SimpleNet(
  (net): Sequential(
    (0): Linear(in_features=1, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_features=32, bias=True)
    (3): ReLU()
    (4): Linear(in_features=32, out_features=1, bias=True)
  )
)


In [3]:
# Training Loop
losses = []
epochs = 100
for epoch in range(epochs):
    model.train()
    epoch_loss = 0.0
    for xb, yb in train_dl:
        pred = model(xb)
        loss = loss_fn(pred, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * xb.size(0)
    avg = epoch_loss / len(train_ds)
    losses.append(avg)
    if (epoch + 1) % 25 == 0:
        print(f'Epoch {epoch+1:3d}/{epochs} | Loss: {avg:.4f}')

Epoch  25/100 | Loss: 0.0911


Epoch  50/100 | Loss: 0.0846


Epoch  75/100 | Loss: 0.0874


Epoch 100/100 | Loss: 0.0846


In [4]:
# Loss Curve
plt.figure(figsize=(8, 4))
plt.plot(losses, color='steelblue', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Training Loss Curve')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('loss_curve.png', dpi=100)
plt.show()
print('Loss curve saved.')

Loss curve saved.


C:\Users\PC\AppData\Local\Temp\ipykernel_2508\2978513522.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [5]:
# Evaluation
model.eval()
with torch.no_grad():
    train_pred = model(x_train_t).squeeze()
    test_pred = model(x_test_t).squeeze()
    train_mse = loss_fn(train_pred, y_train_t.squeeze()).item()
    test_mse = loss_fn(test_pred, y_test_t.squeeze()).item()

print(f'Train MSE: {train_mse:.4f}')
print(f'Test MSE:  {test_mse:.4f}')
print(f'Test RMSE: {test_mse**0.5:.4f}')
print('VERIFICATION PASSED: Phase 17.2 complete')

Train MSE: 0.0814
Test MSE:  0.1696
Test RMSE: 0.4118
VERIFICATION PASSED: Phase 17.2 complete
